# # Install dependencies

In [ ]:
!pip install -q --user scikit-learn torch


# # Load training data

In [ ]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (PyTorch)

In [ ]:
"""Define the model in ``src/iris_net.py`` so ``infer_pytorch.ipynb`` can
unpickle it later. A class defined inside this notebook is pickled as
``__main__.IrisNet`` and won't resolve in another process.
"""
import torch
import torch.nn as nn
from src.iris_net import IrisNet


In [ ]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring;
                    "test" populates `t.X_test / t.y_test`)
    target        — label column name(s); single string or list of strings.
    model_type    — free-form string tagged on the run as `model_type`
                    (surfaced in the nubison UI). "classifier" and
                    "regressor" additionally make `t.fit()` log
                    `val_accuracy` / `val_r2`. None skips the tag.
    weights_path  — where `t.save(model)` writes the pickle.
                    Default "src/weights.pkl" matches `register(artifact_dirs="src")`.

Raw PyTorch loops are an autolog blind spot — mlflow has no hook for vanilla
`nn.Module` training. Hyperparams are recorded explicitly via
`train(extra_params={...})`; per-epoch metrics via `t.log_metric(...)`.
"""
from nubison_model import train

EPOCHS, LR = 30, 0.01

with train(
    datasets=datasets,
    target=["target"],
    model_type="classifier",
    extra_params={
        "model": "IrisNet",
        "optimizer": "Adam",
        "learning_rate": LR,
        "epochs": EPOCHS,
        "loss_fn": "CrossEntropyLoss",
    },
) as t:
    X = torch.tensor(t.X_train.values, dtype=torch.float32)
    y = torch.tensor(t.y_train.values.ravel(), dtype=torch.long)

    model = IrisNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        t.log_metric("train_loss", loss.item(), step=epoch)
        train_acc = (logits.argmax(dim=1) == y).float().mean().item()
        t.log_metric("train_accuracy", train_acc, step=epoch)

    if t.X_val is not None and t.y_val is not None:
        model.eval()
        with torch.no_grad():
            X_val = torch.tensor(t.X_val.values, dtype=torch.float32)
            y_val = t.y_val.values.ravel()
            val_pred = model(X_val).argmax(dim=1).numpy()
            val_acc = (val_pred == y_val).mean()
        t.log_metric("val_accuracy", float(val_acc))

    t.save(model)

print(f"run_id: {t.run_id}")
